# Task 1 - Structured Output: Model Comparison (Bedrock Converse API)

Baseline for every model is **prompt-engineered JSON** on a single email; results flow into a
pandas DataFrame with validity, latency, tokens and cost, plus a scorecard.

**Tool calling is optional and applies only to the one model you set in `TOOL_MODEL`** - see the
tool-vs-prompt cell near the bottom, which compares latency and prints both JSON outputs.

### 1. Install & imports

In [3]:
%pip install boto3 pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [4]:
import re
import json
import time

import boto3
import pandas as pd
from botocore.exceptions import ClientError

### 2. Config
Edit `MODELS` to IDs enabled in your region. `TOOL_MODEL` is the one model the tool-vs-prompt cell uses. `PRICING` values are **placeholders - verify them**.

In [80]:
REGION = "us-east-1"

# Edit to models enabled in your region.
MODELS = [
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "amazon.nova-lite-v1:0",
    "us.meta.llama3-1-8b-instruct-v1:0",
    "google.gemma-3-4b-it",
]


# The ONE model the tool-vs-prompt cell tests with tool calling:
TOOL_MODEL = "amazon.nova-lite-v1:0"

TEMPERATURE = 0.0
MAX_TOKENS = 512

# EXAMPLE prices, USD per 1,000 tokens: (input, output). Verify per region.
# Models not listed here show cost as "n/a".
PRICING = {
    "us.anthropic.claude-haiku-4-5-20251001-v1:0": (0.00100, 0.00500),
    "amazon.nova-lite-v1:0":                       (0.00006, 0.00024),
    "us.meta.llama3-1-8b-instruct-v1:0":          (0.00022, 0.00022),
    "google.gemma-3-4b-it":                        (0.00004, 0.00008),
}

### 3. Email
Single email under test (multi-email dataset removed).

In [82]:
EMAIL_ID = "support_email"

EMAIL = """Subject: Still no refund - running out of patience

My name is Arshaq Kirmani. I returned the wireless headphones three weeks ago and
was promised a refund in 5 business days. I've emailed twice with no reply.
Resolve this today or I'm disputing the charge with my bank.
Kirmani"""

### 4. Prompts, schema & signals

In [84]:
def request_text(email):
    return (
        "Extract the customer's name, their issue, the sentiment "
        "(positive / neutral / negative), and the urgency (low / medium / high) "
        "from this customer support email:\n\n"
        f'"""\n{email}\n"""'
    )

# Prompt path: explicitly demand JSON only.
JSON_ONLY_SYSTEM = (
    "You are a precise information-extraction engine. Respond with a SINGLE "
    "valid JSON object and nothing else - no explanations, no markdown, no "
    "code fences."
)
JSON_ONLY_SUFFIX = (
    '\n\nReturn a single JSON object with exactly these keys: '
    '"name", "issue", "sentiment", "urgency". Output JSON only.'
)

EXPECTED_KEYS = {"name", "issue", "sentiment", "urgency"}
ALLOWED_SENTIMENT = {"positive", "neutral", "negative"}
ALLOWED_URGENCY = {"low", "medium", "high"}

EXTRACTION_TOOL = {
    "toolSpec": {
        "name": "record_ticket_fields",
        "description": "Record the key fields extracted from a customer support email.",
        "inputSchema": {"json": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Customer's full name, or empty if none."},
                "issue": {"type": "string", "description": "Short description of the problem."},
                "sentiment": {"type": "string", "enum": sorted(ALLOWED_SENTIMENT)},
                "urgency": {"type": "string", "enum": sorted(ALLOWED_URGENCY)},
            },
            "required": ["name", "issue", "sentiment", "urgency"],
        }},
    }
}
TOOL_NAME = EXTRACTION_TOOL["toolSpec"]["name"]


class ToolUseUnsupported(Exception):
    """Model rejected toolConfig entirely."""


class ToolNotCalled(Exception):
    """Model accepted tools but replied with text instead of calling one."""

### 5. Extraction paths

In [86]:
def run_prompt(client, model_id, email):
    """Prompt-instructed JSON. Returns (parsed_or_None, meta)."""
    inference = {"maxTokens": MAX_TOKENS, "temperature": TEMPERATURE}
    user_text = request_text(email) + JSON_ONLY_SUFFIX

    start = time.time()
    try:
        resp = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": user_text}]}],
            system=[{"text": JSON_ONLY_SYSTEM}],
            inferenceConfig=inference,
        )
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        msg = e.response["Error"]["Message"].lower()
        if code_ == "ValidationException" and "system" in msg:
            resp = client.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": JSON_ONLY_SYSTEM + "\n\n" + user_text}]}],
                inferenceConfig=inference,
            )
        else:
            raise
    latency = time.time() - start

    # Some models put a reasoning block first - grab the text block, not [0].
    blocks = resp["output"]["message"]["content"]
    raw = next((b["text"] for b in blocks if "text" in b), "")
    meta = {"latency": latency, "usage": resp.get("usage", {}),
            "clean": False, "used_fence": False, "raw": raw.strip()}

    parsed = None
    try:
        parsed = json.loads(raw)
        meta["clean"] = True
    except json.JSONDecodeError:
        if "```" in raw:
            meta["used_fence"] = True
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group(0))
            except json.JSONDecodeError:
                parsed = None
    return parsed, meta


def run_tool(client, model_id, email):
    """Schema-enforced JSON via tool use. Raises ToolUseUnsupported / ToolNotCalled."""
    inference = {"maxTokens": MAX_TOKENS, "temperature": TEMPERATURE}
    messages = [{"role": "user", "content": [{"text": request_text(email)}]}]

    def _call(tool_choice):
        return client.converse(
            modelId=model_id,
            messages=messages,
            toolConfig={"tools": [EXTRACTION_TOOL], "toolChoice": tool_choice},
            inferenceConfig=inference,
        )

    start = time.time()
    try:
        resp = _call({"tool": {"name": TOOL_NAME}})
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        msg = e.response["Error"]["Message"].lower()
        if code_ == "ValidationException":
            if "toolchoice" in msg or "tool choice" in msg:
                resp = _call({"auto": {}})   # e.g. Llama: tools yes, forced choice no
            else:
                raise ToolUseUnsupported(msg) from e
        else:
            raise
    latency = time.time() - start

    meta = {"latency": latency, "usage": resp.get("usage", {})}
    for block in resp["output"]["message"]["content"]:
        if "toolUse" in block:
            return block["toolUse"]["input"], meta
    raise ToolNotCalled()

### 6. Validation & cost

In [88]:
def check(parsed):
    """Return (keys_ok, values_ok, problems[])."""
    if not isinstance(parsed, dict):
        return False, False, ["not a JSON object"]

    missing = EXPECTED_KEYS - set(parsed.keys())
    keys_ok = not missing
    problems = [f"missing {k}" for k in sorted(missing)]

    for field, allowed in (("sentiment", ALLOWED_SENTIMENT), ("urgency", ALLOWED_URGENCY)):
        if field in parsed and str(parsed[field]).strip().lower() not in allowed:
            problems.append(f"{field}={parsed[field]!r} not in {sorted(allowed)}")
    for field in ("name", "issue"):
        if field in parsed and not (isinstance(parsed[field], str) and parsed[field].strip()):
            problems.append(f"{field} empty/non-string")

    values_ok = keys_ok and not [p for p in problems if not p.startswith("missing")]
    return keys_ok, values_ok, problems


def cost(model_id, usage):
    price = PRICING.get(model_id)
    if not price or not usage:
        return None
    in_tok = usage.get("inputTokens", 0)
    out_tok = usage.get("outputTokens", 0)
    return (in_tok / 1000) * price[0] + (out_tok / 1000) * price[1]

### 7. Bedrock client

In [90]:
client = boto3.client("bedrock-runtime", region_name=REGION)
# Optional credential check:
# boto3.client("sts", region_name=REGION).get_caller_identity()

### 8. Run - prompt-engineered JSON for every model
Runs each model on the single email, prints its JSON output, and collects records.
Tool calling is NOT run here - it is isolated to the tool-vs-prompt cell below.

In [92]:
def make_record(model_id, eid, method, parsed, meta):
    keys_ok, values_ok, problems = (
        check(parsed) if parsed is not None else (False, False, ["no JSON parsed"])
    )
    usage = meta.get("usage", {})
    return {
        "model": model_id.split(":")[0],
        "email": eid,
        "method": method,
        "valid_json": parsed is not None,
        "clean": bool(meta.get("clean")) if method == "prompt" else None,
        "values_ok": values_ok,
        "latency_s": round(meta.get("latency", 0.0), 3),
        "in_tok": usage.get("inputTokens", 0),
        "out_tok": usage.get("outputTokens", 0),
        "cost_usd": cost(model_id, usage),
        "problems": "; ".join(problems),
        "parsed": parsed,
    }


records = []

for model_id in MODELS:
    print(f"=== {model_id} ===")
    try:
        parsed, meta = run_prompt(client, model_id, EMAIL)
    except ClientError as e:
        err = e.response["Error"]
        print(f"  unavailable - {err['Code']}: {err['Message']}\n")
        continue
    records.append(make_record(model_id, EMAIL_ID, "prompt", parsed, meta))
    print(f"  ({meta['latency']:.2f}s, clean={meta['clean']})")
    print(json.dumps(parsed, indent=2, ensure_ascii=False) if parsed is not None else meta["raw"])
    print()

print(f"rows: {len(records)}")

=== us.anthropic.claude-haiku-4-5-20251001-v1:0 ===
  (1.96s, clean=False)
{
  "name": "Priya Nair",
  "issue": "Refund not received for returned wireless headphones after three weeks; no response to follow-up emails",
  "sentiment": "negative",
  "urgency": "high"
}

=== amazon.nova-lite-v1:0 ===
  (0.80s, clean=False)
{
  "name": "Priya Nair",
  "issue": "Still no refund after returning wireless headphones three weeks ago.",
  "sentiment": "negative",
  "urgency": "high"
}

=== us.meta.llama3-1-8b-instruct-v1:0 ===
  (0.53s, clean=True)
{
  "name": "Priya Nair",
  "issue": "Refund for returned wireless headphones",
  "sentiment": "negative",
  "urgency": "high"
}

=== google.gemma-3-4b-it ===
  (0.59s, clean=False)
{
  "name": "Priya Nair",
  "issue": "Refund for returned wireless headphones not received",
  "sentiment": "negative",
  "urgency": "high"
}

rows: 4


### 9. Raw results table
Every extraction, including the parsed JSON. Sort/filter as you like.

In [94]:
df = pd.DataFrame(records)
df[["model", "email", "method", "valid_json", "clean", "values_ok",
    "latency_s", "in_tok", "out_tok", "cost_usd", "problems", "parsed"]]

,model,email,method,valid_json,clean,values_ok,latency_s,in_tok,out_tok,cost_usd,problems,parsed
0,us.anthropic.claude-haiku-4-5-20251001-v1,support_email,prompt,True,False,True,1.960,183,69,NaN,,"{'name': 'Priya Nair', 'issue': 'Refund not re..."
1,amazon.nova-lite-v1,support_email,prompt,True,False,True,0.796,159,47,0.000021,,"{'name': 'Priya Nair', 'issue': 'Still no refu..."
2,us.meta.llama3-1-8b-instruct-v1,support_email,prompt,True,True,True,0.531,173,42,0.000047,,"{'name': 'Priya Nair', 'issue': 'Refund for re..."
3,google.gemma-3-4b-it,support_email,prompt,True,False,True,0.594,173,50,NaN,,"{'name': 'Priya Nair', 'issue': 'Refund for re..."


### 10. Scorecard
Aggregated per model + method.

In [96]:
def rate(s):
    s = s.dropna()
    return f"{int(s.sum())}/{len(s)}" if len(s) else "-"

scorecard = (
    df.groupby(["model", "method"])
      .agg(json=("valid_json", rate),
           clean=("clean", rate),
           values=("values_ok", rate),
           avg_latency_s=("latency_s", "mean"),
           total_cost_usd=("cost_usd", lambda s: s.sum(min_count=1)))
      .reset_index()
)
scorecard["avg_latency_s"] = scorecard["avg_latency_s"].round(2)
scorecard["total_cost_usd"] = scorecard["total_cost_usd"].round(5)
scorecard

,model,method,json,clean,values,avg_latency_s,total_cost_usd
0,amazon.nova-lite-v1,prompt,1/1,0/1,1/1,0.80,0.00002
1,google.gemma-3-4b-it,prompt,1/1,0/1,1/1,0.59,NaN
2,us.anthropic.claude-haiku-4-5-20251001-v1,prompt,1/1,0/1,1/1,1.96,NaN
3,us.meta.llama3-1-8b-instruct-v1,prompt,1/1,1/1,1/1,0.53,0.00005


### 11. Tool calling vs prompt - one model only
Runs `TOOL_MODEL` both ways on the same email, prints both JSON outputs, and compares latency.
Tool calling is attempted only here. Change `TOOL_MODEL` to compare a different model.

In [98]:
print(f"TOOL vs PROMPT for: {TOOL_MODEL}\n")

# 1) prompt-engineered
p_parsed, p_meta = run_prompt(client, TOOL_MODEL, EMAIL)

# 2) tool calling (optional; only this model)
tool_ok = True
try:
    t_parsed, t_meta = run_tool(client, TOOL_MODEL, EMAIL)
except ToolUseUnsupported:
    tool_ok = False
    print("This model does not support tool calling - prompt output only.\n")
except ToolNotCalled:
    tool_ok = False
    print("Model offered tools but returned text instead.\n")


def show(tag, parsed, meta):
    _, values_ok, _ = check(parsed) if parsed is not None else (False, False, None)
    c = cost(TOOL_MODEL, meta.get("usage", {}))
    cstr = f"${c:.5f}" if c is not None else "n/a"
    print(f"[{tag}] {meta['latency']:.2f}s  values_ok={values_ok}  cost={cstr}")
    print(json.dumps(parsed, indent=2, ensure_ascii=False) if parsed is not None else meta.get("raw", ""))
    print()

show("prompt", p_parsed, p_meta)
if tool_ok:
    show("tool", t_parsed, t_meta)
    diff = t_meta["latency"] - p_meta["latency"]
    faster = "tool" if diff < 0 else "prompt"
    print(f"Latency delta: {abs(diff):.2f}s  ({faster} faster)")

TOOL vs PROMPT for: amazon.nova-lite-v1:0

[prompt] 0.59s  values_ok=True  cost=$0.00002
{
  "name": "Priya Nair",
  "issue": "Still no refund after returning wireless headphones three weeks ago.",
  "sentiment": "negative",
  "urgency": "high"
}

[tool] 0.73s  values_ok=True  cost=$0.00004
{
  "sentiment": "negative",
  "issue": "Still no refund after returning the wireless headphones",
  "urgency": "high",
  "name": "Priya Nair"
}

Latency delta: 0.14s  (prompt faster)


### 12. Slice & analyze
Starting points - adapt freely.

In [100]:
# Any failure (bad JSON or out-of-domain values), with the reason:
df[~df["values_ok"]][["model", "email", "method", "problems"]]

,model,email,method,problems


In [101]:
# Per-model value validity (prompt baseline):
(df.assign(ok=df["values_ok"].astype(int))
   .pivot_table(index="model", columns="method", values="ok", aggfunc="mean"))

method,prompt
model,
amazon.nova-lite-v1,1.0
google.gemma-3-4b-it,1.0
us.anthropic.claude-haiku-4-5-20251001-v1,1.0
us.meta.llama3-1-8b-instruct-v1,1.0


In [102]:
# Inspect one actual extraction:
row = df[df["method"] == "prompt"].iloc[0]
print(row["model"])
row["parsed"]

us.anthropic.claude-haiku-4-5-20251001-v1


{'name': 'Priya Nair',
 'issue': 'Refund not received for returned wireless headphones after three weeks; no response to follow-up emails',
 'sentiment': 'negative',
 'urgency': 'high'}